In [13]:
import numpy as np
from scipy.optimize import minimize_scalar, brentq
from scipy.stats import norm


def estimar_lomax(T, confianza=0.95):
    """
    Estimación MLE de underline{rho} y sigma_rho para la distribución

        f(T) = underline_rho *
               (1 + sigma_rho*T)^(-(underline_rho/sigma_rho + 1))

    y cálculo de intervalos de confianza de Wald.

    Parámetros
    ----------
    T : array-like
        Datos observados T_i.
    confianza : float
        Nivel de confianza. Por defecto 0.95.

    Retorna
    -------
    resultados : dict
        Diccionario con estimaciones, errores estándar,
        intervalos de confianza, matriz de información
        y matriz de covarianza.
    """

    # ---------------------------------------------------------
    # 1. Preparar datos
    # ---------------------------------------------------------

    T = np.asarray(T, dtype=float)

    if T.ndim != 1:
        raise ValueError("T debe ser un arreglo unidimensional.")

    if len(T) < 2:
        raise ValueError("Se necesitan al menos dos observaciones.")

    if np.any(~np.isfinite(T)):
        raise ValueError("T contiene valores no finitos.")

    if np.any(T < 0):
        raise ValueError("Todos los valores T_i deben ser >= 0.")

    n = len(T)

    # ---------------------------------------------------------
    # 2. Log-verosimilitud perfilada
    # ---------------------------------------------------------
    #
    # Para un sigma dado:
    #
    # rho_hat(sigma) =
    #       n*sigma / sum log(1 + sigma*T_i)
    #
    # ---------------------------------------------------------

    def S(sigma):
        return np.sum(np.log1p(sigma * T))

    def rho_hat_sigma(sigma):
        s = S(sigma)

        if s <= 0:
            return np.inf

        return n * sigma / s

    def loglik_profile(sigma):
        """
        Log-verosimilitud después de maximizar
        analíticamente respecto a underline_rho.
        """

        if sigma <= 0:
            return -np.inf

        r = rho_hat_sigma(sigma)
        s = S(sigma)

        return n * np.log(r) - (r / sigma + 1.0) * s

    # ---------------------------------------------------------
    # 3. Encontrar sigma_hat
    # ---------------------------------------------------------
    #
    # Se optimiza sobre log(sigma) para garantizar sigma > 0.
    # ---------------------------------------------------------

    def objective_log_sigma(log_sigma):
        sigma = np.exp(log_sigma)
        return -loglik_profile(sigma)

    # Intervalo inicial amplio para log(sigma)
    resultado_opt = minimize_scalar(
        objective_log_sigma,
        bounds=(-20, 20),
        method="bounded",
        options={"xatol": 1e-12}
    )

    if not resultado_opt.success:
        raise RuntimeError(
            "No fue posible encontrar el MLE de sigma_rho."
        )

    sigma_hat = np.exp(resultado_opt.x)

    # ---------------------------------------------------------
    # 4. Obtener rho_hat
    # ---------------------------------------------------------

    rho_hat = rho_hat_sigma(sigma_hat)

    # ---------------------------------------------------------
    # 5. Calcular cantidades necesarias para la información
    # ---------------------------------------------------------

    S_hat = S(sigma_hat)

    A_hat = np.sum(
        T / (1.0 + sigma_hat * T)
    )

    B_hat = np.sum(
        T**2 / (1.0 + sigma_hat * T)**2
    )

    # ---------------------------------------------------------
    # 6. Matriz de información observada
    # ---------------------------------------------------------

    J11 = n / rho_hat**2

    J12 = (
        A_hat / sigma_hat
        - S_hat / sigma_hat**2
    )

    J22 = (
        -2.0 * rho_hat * A_hat / sigma_hat**2
        + 2.0 * rho_hat * S_hat / sigma_hat**3
        - (rho_hat / sigma_hat + 1.0) * B_hat
    )

    J = np.array([
        [J11, J12],
        [J12, J22]
    ])

    # ---------------------------------------------------------
    # 7. Matriz de covarianza
    # ---------------------------------------------------------

    try:
        cov = np.linalg.inv(J)
    except np.linalg.LinAlgError:
        raise RuntimeError(
            "La matriz de información es singular."
        )

    # ---------------------------------------------------------
    # 8. Errores estándar
    # ---------------------------------------------------------

    var_rho = cov[0, 0]
    var_sigma = cov[1, 1]

    if var_rho <= 0 or var_sigma <= 0:
        raise RuntimeError(
            "La matriz de covarianza no es positiva definida."
        )

    se_rho = np.sqrt(var_rho)
    se_sigma = np.sqrt(var_sigma)

    # ---------------------------------------------------------
    # 9. Intervalos de confianza de Wald
    # ---------------------------------------------------------

    alpha = 1.0 - confianza
    z = norm.ppf(1.0 - alpha / 2.0)

    ci_rho = (
        rho_hat - z * se_rho,
        rho_hat + z * se_rho
    )

    ci_sigma = (
        sigma_hat - z * se_sigma,
        sigma_hat + z * se_sigma
    )

    # ---------------------------------------------------------
    # 10. Correlación entre los estimadores
    # ---------------------------------------------------------

    correlation = (
        cov[0, 1]
        / np.sqrt(cov[0, 0] * cov[1, 1])
    )

    # ---------------------------------------------------------
    # 11. Verificación de las ecuaciones MLE
    # ---------------------------------------------------------

    score_rho = (
        n / rho_hat
        - S_hat / sigma_hat
    )

    score_sigma = (
        rho_hat * S_hat / sigma_hat**2
        - (rho_hat / sigma_hat + 1.0) * A_hat
    )

    # ---------------------------------------------------------
    # 12. Resultados
    # ---------------------------------------------------------

    resultados = {
        "n": n,

        "rho_hat": rho_hat,
        "sigma_hat": sigma_hat,

        "se_rho": se_rho,
        "se_sigma": se_sigma,

        "ci_rho": ci_rho,
        "ci_sigma": ci_sigma,

        "confidence": confianza,
        "z": z,

        "S": S_hat,
        "A": A_hat,
        "B": B_hat,

        "information_matrix": J,
        "covariance_matrix": cov,

        "correlation": correlation,

        "log_likelihood": loglik_profile(sigma_hat),

        "score_rho": score_rho,
        "score_sigma": score_sigma
    }

    return resultados

In [14]:
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import lomax
from scipy.stats import kstest

archivo = "Analisis_GATOS_LISTA2.xlsx"

df = pd.read_excel(archivo, sheet_name="Duraciones")

# Duraciones cuando el estado es ACTIVO
T = df.loc[df["Estado"] == "ACTIVO", "Duracion_min"].dropna().to_numpy()

# Duraciones cuando el estado es INACTIVO
inactivo = df.loc[df["Estado"] == "INACTIVO", "Duracion_min"].dropna().to_numpy()

In [15]:
resultado = estimar_lomax(T)

In [16]:
print("rho_hat   =", resultado["rho_hat"])
print("sigma_hat =", resultado["sigma_hat"])

rho_hat   = 0.3129312650655362
sigma_hat = 0.18260071748631573


In [17]:
print("SE(rho)   =", resultado["se_rho"])
print("SE(sigma) =", resultado["se_sigma"])

SE(rho)   = 0.04361370649175647
SE(sigma) = 0.06044467884295539


In [18]:
print("IC rho   =", resultado["ci_rho"])
print("IC sigma =", resultado["ci_sigma"])

IC rho   = (np.float64(0.22744997110939277), np.float64(0.3984125590216796))
IC sigma = (np.float64(0.064131323897033), np.float64(0.30107011107559845))


In [19]:
print(resultado["rho_hat"]/resultado["sigma_hat"])

1.7137460869450722


In [23]:
from scipy.special import hyp2f1
l2= 1.7137460869450722
l1=5.386270358968124
l3=0.1826/0.0302607
l4=hyp2f1(1, l2, l1+l2+1, 1-l3)

l5= l1*l4/(l1+l2)


print(l4,l5)

0.5360531672609881 0.40666487291178055
